# Part 1

## Task 1

In [207]:
import pandas as pd

loans_raw = pd.read_csv("../loans.csv")

Inital size on loans. (Expected to be: (623, 9))

In [208]:
loans = loans_raw.copy()

loans.shape

(623, 9)

Outliars in the data. (Expected: [notes have all rows to be null, dates and fee are represented as str and not date and something numerical, return date has 30 missing (593 compared to 623)])

In [209]:
loans.info()

<class 'pandas.DataFrame'>
RangeIndex: 623 entries, 0 to 622
Data columns (total 9 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   loan_id      623 non-null    str    
 1   member_id    623 non-null    str    
 2   title        623 non-null    str    
 3   genre        623 non-null    str    
 4   loan_date    623 non-null    str    
 5   return_date  593 non-null    str    
 6   late_fee     623 non-null    str    
 7   returned     623 non-null    str    
 8   notes        0 non-null      float64
dtypes: float64(1), str(8)
memory usage: 43.9 KB


## Task 2

How many genres there are (without filtering). (Should be 28 genres)

In [210]:
loans['genre'].unique()

<StringArray>
[       ' ROMANCE',           'Crime',         'fantasy',           'CRIME',
       'Children ',        'children',          'Crime ',      'children's',
         'history',         'romance',          'sci-fi',        'Children',
         'Fantasy',          'Sci-fi',        'History ',         'Romance',
 'Science fiction',          'Sci fi',           'crime',        'Romance ',
         'FANTASY',          ' crime',        ' Fantasy',         'History',
        ' history',      'Children's',         'sci fi ', 'science fiction']
Length: 28, dtype: str

How many genres there are (after filtering). (Should be 9 genres)

In [211]:
loans['genre'] = loans['genre'].str.strip().str.lower()

loans['genre'].unique()

<StringArray>
[        'romance',           'crime',         'fantasy',        'children',
      'children's',         'history',          'sci-fi', 'science fiction',
          'sci fi']
Length: 9, dtype: str

How many genres there are (after replacing). (Should be 6)

In [212]:
loans['genre'] = (
   loans['genre']
   .str.replace("'", "", regex=False)
   .str.replace("-", " ", regex=False)
   .str.title()
   .str.replace({"Sci Fi": "Science Fiction", "Childrens" : "Children"})
)
loans['genre'].unique()

<StringArray>
['Romance', 'Crime', 'Fantasy', 'Children', 'History', 'Science Fiction']
Length: 6, dtype: str

Loans per genre after extensive cleaning. (Should be [Crime - 163, Children - 135, Fantasy - 90, Sci-Fi 90, Romance - 86, History - 59])

In [213]:
loans['genre'].value_counts()

genre
Crime              163
Children           135
Fantasy             90
Science Fiction     90
Romance             86
History             59
Name: count, dtype: int64

## Task 3

Unique return values before any change. (Should be 10)

In [214]:
#loans['returned'].nunique()
loans['returned'].unique()

<StringArray>
['Yes', 'YES', 'Y', 'y', 'yes', 'No', 'N', 'n', 'NO', 'no']
Length: 10, dtype: str

Count per category after filtering. (Should be 593 on True and 30 on False)

In [215]:
loans['returned'] = loans['returned'].str.strip().str.lower().replace({"y" : "yes", "n" : "no"}).map({'yes': True, 'no': False})
loans['returned'].value_counts()

returned
True     593
False     30
Name: count, dtype: int64

Check of no miss. (Should be 0)

In [216]:
print(loans['returned'].isna().sum())

0


## Task 4

Float conversion on the "late_fee" column should cause a ValueError.

In [217]:
loans["late_fee"].astype(float)

ValueError: could not convert string to float: '7,50 kr'

Filtering steps.

In [218]:
loans["late_fee"] = loans["late_fee"].str.replace("kr", "")
print(loans["late_fee"])

loans["late_fee"] = loans["late_fee"].str.strip()
print(loans["late_fee"])

loans["late_fee"] = loans["late_fee"].str.replace(",", ".", regex=False)
print(loans["late_fee"])

0        7,50 
1        0,00 
2        0,00 
3        0,00 
4        0,00 
        ...   
618      0,00 
619      25,00
620     40,00 
621      0,00 
622      0,00 
Name: late_fee, Length: 623, dtype: str
0       7,50
1       0,00
2       0,00
3       0,00
4       0,00
       ...  
618     0,00
619    25,00
620    40,00
621     0,00
622     0,00
Name: late_fee, Length: 623, dtype: str
0       7.50
1       0.00
2       0.00
3       0.00
4       0.00
       ...  
618     0.00
619    25.00
620    40.00
621     0.00
622     0.00
Name: late_fee, Length: 623, dtype: str


Working float conversion post-filtering.

In [219]:
loans["late_fee"] = loans["late_fee"].astype(float)

Should now give the following values: [min - 0.0, max - 47.5, mean - 12.3, sum/total - 7677.5]

In [220]:
print(loans["late_fee"].describe().round(1))
first_sum = loans["late_fee"].sum()
print(first_sum)

count    623.0
mean      12.3
std       16.1
min        0.0
25%        0.0
50%        0.0
75%       25.0
max       47.5
Name: late_fee, dtype: float64
7677.5


## Task 5

In [221]:
loans["loan_date"] = pd.to_datetime(loans["loan_date"], errors="coerce")

How many that are NaT. (Should be 3)

In [222]:
print(loans["loan_date"].isna().sum())

3


First and last date within the "loan_date" column. (Should be: loans['loan_date'])

In [223]:
print(loans['loan_date'].min())
print(loans['loan_date'].max())

2026-01-05 00:00:00
2026-06-27 00:00:00


Loans per month. (Should be: [January - 96, February - 100, March - 111, April - 106, May - 108, June - 99])

In [224]:
loans_per_month = loans['loan_date'].dt.to_period('M').value_counts().sort_index()
print(loans_per_month)
print(loans_per_month.sum())

loan_date
2026-01     96
2026-02    100
2026-03    111
2026-04    106
2026-05    108
2026-06     99
Freq: M, Name: count, dtype: int64
620


The sum is 620, and that plus the NaT ones (3) = 623. Same answer as the expected answer.

## Task 6

how many duplicates there are in the raw table.

Duplicates in raw version. (Should be 12)

In [225]:
print(loans_raw.duplicated().sum())

12


Duplicates in cleaned version. (Should be 18)

In [226]:
print(loans.duplicated().sum())

18


In [227]:
print(loans.duplicated(subset='loan_id').sum())

18


They differ in sum because the values have been filtered so that same categories of different spellings are now the same, and so it will find more duplicates as more parts are now matching.

Duplicate removal and index reset.

In [228]:
loans = loans.drop_duplicates().reset_index(drop=True)
loans.shape #(605, 9)

(605, 9)

Increase to the total of the late fees, in kr and precent.
(Should be 230.kr, ~= 3.0%)

Earlier, we got 7677.5, and now we're getting 7447.5. We're getting a price reduction of 230.0.
Should be 3% loss.

In [234]:
second_sum = loans["late_fee"].sum()
sum_difference = first_sum - second_sum
print(sum_difference)
print(f"{(sum_difference/first_sum).round(2)*100}%")

230.0
3.0%


## Task 7

Deleting the empty column "notes".

In [237]:
loans = loans.drop(columns="notes")

Save into a new, cleaned .csv file as well. This will work many times over, because we are resetting loans from its original state through copying loans_raw, and then all steps can be done over and over. The last step - where we store into a .csv file - will just write the new results each time.

In [239]:
loans = loans_raw.copy()

loans['genre'] = loans['genre'].str.strip().str.lower()

loans['genre'] = (
   loans['genre']
   .str.replace("'", "", regex=False)
   .str.replace("-", " ", regex=False)
   .str.title()
   .str.replace({"Sci Fi": "Science Fiction", "Childrens" : "Children"})
)

loans['returned'] = loans['returned'].str.strip().str.lower().replace({"y" : "yes", "n" : "no"}).map({'yes': True, 'no': False})

loans["late_fee"] = loans["late_fee"].str.replace("kr", "")
loans["late_fee"] = loans["late_fee"].str.strip()
loans["late_fee"] = loans["late_fee"].str.replace(",", ".", regex=False)

loans["late_fee"] = loans["late_fee"].astype(float)

loans["loan_date"] = pd.to_datetime(loans["loan_date"], errors="coerce")

loans = loans.drop_duplicates().reset_index(drop=True)

print(loans.shape)

loans.to_csv("../loans_clean.csv", index=False)

(605, 9)


# Part 2

## Task 1

Both cases of missing values for return_date and returned give 30. This should mean that no borrowed books are returned, simply put.

In [ ]:
loans = loans_raw.copy()

loans['genre'] = loans['genre'].str.strip().str.lower()

loans['genre'] = (
   loans['genre']
   .str.replace("'", "", regex=False)
   .str.replace("-", " ", regex=False)
   .str.title()
   .str.replace({"Sci Fi": "Science Fiction", "Childrens" : "Children"})
)

loans['returned'] = loans['returned'].str.strip().str.lower().replace({"y" : "yes", "n" : "no"}).map({'yes': True, 'no': False})

loans["late_fee"] = loans["late_fee"].str.replace("kr", "")
loans["late_fee"] = loans["late_fee"].str.strip()
loans["late_fee"] = loans["late_fee"].str.replace(",", ".", regex=False)

loans["late_fee"] = loans["late_fee"].astype(float)

loans["loan_date"] = pd.to_datetime(loans["loan_date"], errors="coerce")

#Changes right here.
loans["return_date"] = pd.to_datetime(loans["return_date"], errors="coerce")
print(loans["return_date"].isna().sum())
#Compare to "returned".
print((loans["returned"] == False).sum())

loans = loans.drop_duplicates().reset_index(drop=True)

print(loans.shape)

loans.to_csv("../loans_clean.csv", index=False)

30
30
(605, 9)


: 